# Attack the Arduino Uno target

Voltage-glitch the Uno running `glitch_target.ino` while it counts to 10000,
and read its serial result to classify **normal / fault / crash** — the
SimpleLink-FI notebook-5 workflow, on our own target.

Wire it per `arduino/README.md` (D7→trigger, TX/RX→CH1/CH0, crowbar→VCC,
common GND, mind the 5 V↔3.3 V levels). Watch the scope on D7 (trigger) +
the crowbar output to see the glitch land inside the loop.

> ⚠️ Crowbar shorts the Uno's VCC. Start gentle (LP, small width) and add a
> current-limit if unsure. A robust 5 V Uno mostly resets (crash); real
> faults need a tuned delay/width sweep (and maybe decap removal).

In [ ]:
import faultycat as fc

# --- target: the Arduino sketch ---
CMD      = b'\xAA'                 # runs the loop + raises the trigger
EXPECTED = (10000).to_bytes(4, 'little')   # b"\x10'\x00\x00" — un-glitched result
BAUD     = 115200
# --- glitch ---
OUTPUT   = 'lp'
WIDTHS   = range(100, 600, 100)   # ns
DELAYS   = range(0, 2000, 50)     # us from trigger into the loop
REPEATS  = 3
SIM      = False

cat = fc.connect(simulator=SIM)
if cat.uart:
    cat.uart.open(baud=BAUD)
cat.crowbar

In [ ]:
def classify(ret: bytes) -> str:
    if ret == EXPECTED:
        return 'normal'
    if not ret:
        return 'crash'
    return 'success'      # corrupted count = fault injected

In [ ]:
gc = fc.GlitchController(['delay', 'width'], groups=['success', 'crash', 'normal'])
gc.set_range('delay', DELAYS).set_range('width', [w for w in WIDTHS for _ in range(REPEATS)])

for p in gc.glitch_values():
    cat.crowbar.trigger  = 'ext_rising'
    cat.crowbar.output   = OUTPUT
    cat.crowbar.delay_us = p['delay']
    cat.crowbar.width_ns = p['width']
    try:
        cat.crowbar.arm()
        cat.uart.reset_input()
        cat.crowbar.fire(trigger_timeout_ms=500)  # arms the trigger wait
        cat.uart.write(CMD)                        # Uno runs loop -> raises D7 -> glitch
        ret = cat.uart.read(4)                     # 4-byte result (or short/empty)
        gc.add(classify(ret), result=ret.hex())
    except fc.EngineError as e:
        gc.add('crash', result=str(e))
    finally:
        cat.crowbar.disarm()

gc.counts()

In [ ]:
df = gc.results_df()
total = len(df)
for g in ('success', 'crash'):
    n = int((df.group == g).sum())
    print(f'{g:8s}: {n:4d} ({100*n/total:.1f}%)')
fc.glitch_map(df, 'group', x='delay', y='width');

In [ ]:
cat.close()